## Влияния зависимости PARL (представленность женщин в парламенте) на GMI (индекс милитаризации)

### 0. Библиотеки

In [ ]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.stats.outliers_influence import variance_inflation_factor

from linearmodels.iv import IV2SLS

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


import plotly.graph_objects as go
import plotly.express as px

In [ ]:
import warnings
warnings.filterwarnings("ignore")

### 1. Данные

#### 1.1 Источник

In [ ]:
file_path = './phd_data/final_data.csv'

df = pd.read_csv(file_path, sep = ';', decimal = ',')
df.head()

#### 1.2 Смотрим

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# кол-во пропусков
print(df.isnull().sum())

In [ ]:
print(df[['GMI', 'PARL']].describe())

In [ ]:
# корреляционная матрица
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
corr_matrix = df[num_cols].corr()

In [ ]:
fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu',
    title='Корреляции между числовыми переменными'
)
fig.show()

In [ ]:
# Переменные для анализа
features = ['GMI', 'PARL', 'GDP', 'Political Stability', 'Rule of Law',
            'Fertility', 'Corruption']

# Проверяем, сколько уникальных значений у религии
print(df['Religion'].value_counts())

In [ ]:
# Scatter Matrix
fig = px.scatter_matrix(
    df.dropna(subset=features + ['Religion']),
    dimensions=features,
    color='Religion',
    title='Scatter Matrix: основные переменные по религии',
    height=1000
)
fig.update_traces(diagonal_visible=False)
fig.show()

### 2. Модели

#### 2.1 Простая модель

In [ ]:
# Простая модель GMI ~ PARL
df_model = df.dropna(subset=['GMI', 'PARL'])

X = sm.add_constant(df_model[['PARL']])
y = df_model['GMI']

model_simple = sm.OLS(y, X).fit()
print(model_simple.summary())

#### 2.2 Множественная модель

In [ ]:
# Убираем пропуски
df_model = df.dropna(subset=['GMI', 'PARL', 'GDP', 'Political Stability', 'Rule of Law', 'Religion'])

# Числовые переменные
X_num = df_model[['PARL', 'GDP', 'Political Stability', 'Rule of Law']]

# Дамми по религии
X_cat = pd.get_dummies(df_model['Religion'], drop_first=True)

# Собираем всё вместе
X = pd.concat([X_num, X_cat], axis=1)

# Добавляем константу
X = sm.add_constant(X)

# Важно: преобразуем все в float
X = X.astype(float)

y = df_model['GMI']

# Строим модель
model_multi_min = sm.OLS(y, X).fit()

print(model_multi_min.summary())


#### 2.3 Множественная модель +

In [ ]:
# Убираем пропуски по всем нужным переменным
df_model = df.dropna(subset=[
    'GMI', 'PARL', 'GDP', 'Political Stability', 'Rule of Law',
    'Fertility', 'Corruption', 'Women Business', 'Religion'
])

# Числовые переменные
X_num = df_model[['PARL', 'GDP', 'Political Stability', 'Rule of Law',
                  'Fertility', 'Corruption', 'Women Business']]

# Дамми-переменные по религии
X_cat = pd.get_dummies(df_model['Religion'], drop_first=True)

# Объединяем
X = pd.concat([X_num, X_cat], axis=1)

# Добавляем константу и приводим к float
X = sm.add_constant(X).astype(float)
y = df_model['GMI']

# Строим модель
model_full = sm.OLS(y, X).fit()
print(model_full.summary())

### 3. Кластеризация

#### 3.? Базовая кластеризация на 4-х классах (МОЖЕТ ПО ПЕРЦЕНТИЛЯМ)

In [ ]:
# Выбираем признаки для кластеризации
cluster_features = ['GDP', 'Rule of Law', 'Political Stability',
                    'Fertility', 'Life Expectancy', 'Corruption', 'Women Business']

# Берем последние данные по каждой стране
df_latest = df.dropna(subset=cluster_features).sort_values('Year').drop_duplicates('Country Name', keep='last')

# Масштабируем признаки
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_latest[cluster_features])

# Кластеризация на 4 группы (можно поменять)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_latest['cluster'] = kmeans.fit_predict(X_scaled)

In [ ]:
# Посмотрим количество стран в каждом кластере
print(df_latest['cluster'].value_counts())

In [ ]:
# Список стран по каждому кластеру
for cl in sorted(df_latest['cluster'].unique()):
    print(f'Кластер {cl}')
    print(df_latest[df_latest['cluster'] == cl]['Country Name'].tolist())
    print('\n')

In [ ]:
# Средние значения признаков по кластерам
print(df_latest.groupby('cluster')[cluster_features].agg(['min', 'median', 'mean', 'max']))

In [ ]:
# Добавляем кластеры в общий df для построения моделей
df = df.merge(df_latest[['Country Name', 'cluster']], on='Country Name')

In [ ]:
# Используем те же признаки, что при кластеризации
cluster_features = ['GDP', 'Rule of Law', 'Political Stability',
                    'Fertility', 'Life Expectancy', 'Corruption', 'Women Business']

# Забираем только страны с этими признаками
df_latest = df.dropna(subset=cluster_features).sort_values('Year').drop_duplicates('Country Name', keep='last')

X_scaled = scaler.fit_transform(df_latest[cluster_features])

# PCA до 2 компонент
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

df_latest['pca_1'] = components[:, 0]
df_latest['pca_2'] = components[:, 1]

# Визуализация
fig = px.scatter(
    df_latest,
    x='pca_1',
    y='pca_2',
    color=df_latest['cluster'].astype(str),
    hover_name='Country Name',
    title='PCA Scatter: Разделение стран по кластерам'
)

fig.show()

# Можно посмотреть сколько инфы объясняют компоненты
print(f'Explained variance: {pca.explained_variance_ratio_}')

In [ ]:
# Тут задаешь название для каждого кластера руками:
cluster_names = {
    0: 'Россия + восточная Европа',
    1: 'бедная Африка, Иран, Пакистан',
    2: 'западная Европа',
    3: 'Китай + США'
}

df_latest['cluster_name'] = df_latest['cluster'].map(cluster_names)

In [ ]:
# Сохраняем в csv
df_latest[['Country Name', 'cluster', 'cluster_name']].to_csv('country_clusters.csv', index=False)

#### 3.? Альнернативная кластеризация на 4-х классах

In [ ]:
# Выбираем альтернативный набор признаков
alt_features = ['Rule of Law', 'Political Stability', 'Corruption']

# Берем последние данные по каждой стране
df_latest_alt = df.dropna(subset=alt_features).sort_values('Year').drop_duplicates('Country Name', keep='last')

# Масштабируем признаки
scaler_alt = StandardScaler()
X_scaled_alt = scaler_alt.fit_transform(df_latest_alt[alt_features])

# Кластеризация (пока оставим 4 кластера — для сравнения)
kmeans_alt = KMeans(n_clusters=4, random_state=42, n_init=10)
df_latest_alt['cluster_alt'] = kmeans_alt.fit_predict(X_scaled_alt)

In [ ]:
# Сколько стран в каждом новом кластере
print(df_latest_alt['cluster_alt'].value_counts())

In [ ]:
for cl in sorted(df_latest_alt['cluster_alt'].unique()):
    print(f'Кластер {cl}')
    print(df_latest_alt[df_latest_alt['cluster_alt'] == cl]['Country Name'].tolist())
    print('\n')

In [ ]:
pca_alt = PCA(n_components=2)
components_alt = pca_alt.fit_transform(X_scaled_alt)

df_latest_alt['pca_1'] = components_alt[:, 0]
df_latest_alt['pca_2'] = components_alt[:, 1]

# Визуализация
fig = px.scatter(
    df_latest_alt,
    x='pca_1',
    y='pca_2',
    color=df_latest_alt['cluster_alt'].astype(str),
    hover_name='Country Name',
    title='PCA Scatter: Альтернативная кластеризация'
)
fig.show()

print(f'Explained variance: {pca_alt.explained_variance_ratio_}')

In [ ]:
# Тут задаешь название для каждого кластера руками:
cluster_names = {
    0: 'бедная Африка',
    1: 'южная Европа, ОАЭ, Саудовская Аравия',
    2: 'западная Европа',
    3: 'мелкие страны: Бразилия, Молдова, ЮАР'
}

df_latest_alt['cluster_name'] = df_latest_alt['cluster_alt'].map(cluster_names)

In [ ]:
# Сохраняем в csv
df_latest_alt[['Country Name', 'cluster_alt', 'cluster_name']].to_csv('country_clusters_alt.csv', index=False)

#### 3.? Расчет оптимального кол-ва кластеров

In [ ]:
inertia = []
silhouette = []
K_range = range(2, 8)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled_alt)  # или X_scaled для другой кластеризации
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(X_scaled_alt, kmeans.labels_))

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(K_range), y=inertia, mode='lines+markers', name='Inertia'))
fig.add_trace(go.Scatter(x=list(K_range), y=silhouette, mode='lines+markers', name='Silhouette Score'))
fig.update_layout(title='Elbow & Silhouette для определения числа кластеров', xaxis_title='Количество кластеров')
fig.show()

### 4. Простые модели по кластерам

In [ ]:
# Основной датасет
df = pd.read_csv('./phd_data/final_data.csv', sep=';', decimal=',')

# Признаки для моделей
model_features = ['PARL', 'GDP', 'Political Stability', 'Rule of Law', 'Fertility', 'Corruption', 'Women Business']

# Датафрейм для моделей
df_model = df[(df['Year'] >= 1997)]
df_model = df_model.dropna(subset=['GMI'] + model_features)

#### 4.1 Кластеризация 1

In [ ]:
cluster_features = ['GDP', 'Rule of Law', 'Political Stability', 'Fertility',
                    'Life Expectancy', 'Corruption', 'Women Business']

scaler_1 = StandardScaler()
X_1 = scaler_1.fit_transform(df_model.groupby('Country Name')[cluster_features].mean())

kmeans_1 = KMeans(n_clusters=4, random_state=42)
clusters_1 = kmeans_1.fit_predict(X_1)

df_clusters_1 = pd.DataFrame({'Country Name': df_model['Country Name'].unique(), 'cluster_1': clusters_1})
df_model = df_model.merge(df_clusters_1, on='Country Name', how='left')

#### 4.2 Кластеризация 2 (по институтам)

In [ ]:
alt_features = ['Rule of Law', 'Political Stability', 'Corruption']

scaler_2 = StandardScaler()
X_2 = scaler_2.fit_transform(df_model.groupby('Country Name')[alt_features].mean())

kmeans_2 = KMeans(n_clusters=4, random_state=42)
clusters_2 = kmeans_2.fit_predict(X_2)

df_clusters_2 = pd.DataFrame({'Country Name': df_model['Country Name'].unique(), 'cluster_2': clusters_2})
df_model = df_model.merge(df_clusters_2, on='Country Name', how='left')

#### 4.4 Сравнение моделей

In [ ]:
def run_models(df, cluster_col):
    results = []

    for cl in sorted(df[cluster_col].unique()):
        df_cl = df[df[cluster_col] == cl]
        X = df_cl[model_features]
        X = sm.add_constant(X)
        y = df_cl['GMI']

        model = sm.OLS(y, X).fit()

        results.append({
            'cluster': cl,
            'coef': model.params['PARL'],
            'p_value': model.pvalues['PARL'],
            'r_squared': model.rsquared,
            'std_err': model.bse['PARL']
        })

    return pd.DataFrame(results)

In [ ]:
results_1 = run_models(df_model, 'cluster_1')
results_2 = run_models(df_model, 'cluster_2')

In [ ]:
display(results_1)

In [ ]:
display(results_2)

In [ ]:
fig1 = px.bar(results_1, x='cluster', y='coef', error_y='std_err',
              title='Коэффициенты PARL по кластерам (кластеризация экономика + институты + демография)')

fig2 = px.bar(results_2, x='cluster', y='coef', error_y='std_err',
              title='Коэффициенты PARL по кластерам (кластеризация только институты)')

fig1.show()
fig2.show()

### 5. Множественная регрессия по кластерам (институты) + религия

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

# Подтягиваем кластеры по институтам
clusters = pd.read_csv('country_clusters_alt.csv')
df = df.merge(clusters, on='Country Name')

# Создаём дамми-переменные по религии
df = pd.get_dummies(df, columns=['Religion'], drop_first=True)

# Переменные модели
base_controls = ['PARL', 'GDP', 'Political Stability', 'Rule of Law',
                 'Fertility', 'Corruption', 'Women Business']

religion_dummies = [col for col in df.columns if col.startswith('Religion_')]

controls = base_controls + religion_dummies

# Берём нужные года и данные без пропусков
df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI'] + controls)

# Модели по кластерам
results_multi = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Конвертируем все X в float
    X = sm.add_constant(df_cl[controls].astype(float))
    y = df_cl['GMI']

    model = sm.OLS(y, X).fit()
    print(f"Кластер {cl}")
    print(model.summary())

    coef = model.params['PARL']
    p_value = model.pvalues['PARL']
    r_squared = model.rsquared
    std_err = model.bse['PARL']

    results_multi.append({
        'cluster_name': cl,
        'coef': coef,
        'p_value': round(p_value,4),
        'r_squared': r_squared,
        'std_err': std_err
    })

results_multi_df = pd.DataFrame(results_multi)
display(results_multi_df)

In [ ]:
# График коэффициентов
fig = px.bar(
    results_multi_df,
    x='cluster_name',
    y='coef',
    error_y='std_err',
    title='Коэффициенты PARL в моделях по кластерам (множ. модель + религия)',
    labels={'coef': 'Коэффициент при PARL', 'cluster': 'Кластер'}
)

fig.show()

#### 6. Множественная регрессия по кластерам (институты) + Is_muslim

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

# Кластеры по институтам
# кластеризацию сюда сразу руками вставляем
clusters = pd.read_csv('country_clusters_alt.csv')
df = df.merge(clusters, on='Country Name')

# Новый бинарный признак — мусульманская страна или нет
df['Is_muslim'] = df['Religion'].apply(lambda x: 1 if x == 'Muslim' else 0)

# Контрольные переменные
controls = ['PARL', 'GDP', 'Political Stability', 'Rule of Law',
            'Fertility', 'Corruption', 'Women Business', 'Is_muslim']

# Только нужные года и строки без пропусков
df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI'] + controls)

results_multi = []

# Прогоняем по каждому кластеру
for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    X = sm.add_constant(df_cl[controls])
    y = df_cl['GMI']

    model = sm.OLS(y, X).fit()
    print(f"Кластер {cl}")
    print(model.summary())

    coef = model.params['PARL']
    p_value = model.pvalues['PARL']
    r_squared = model.rsquared
    std_err = model.bse['PARL']

    results_multi.append({
        'cluster_name': cl,
        'coef': coef,
        'p_value': round(p_value,4),
        'r_squared': r_squared,
        'std_err': std_err
    })

results_multi_df = pd.DataFrame(results_multi)
display(results_multi_df)

#### 7. Множественная регрессия по кластерам (институты) без религии

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

# Подтягиваем кластеры по институтам
clusters = pd.read_csv('country_clusters_alt.csv')
df = df.merge(clusters, on='Country Name')

# Переменные модели
controls = ['PARL', 'GDP', 'Political Stability', 'Rule of Law',
                 'Fertility', 'Corruption', 'Women Business']

# Берём нужные года и данные без пропусков
df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI'] + controls)

# Модели по кластерам
results_multi = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Конвертируем все X в float
    X = sm.add_constant(df_cl[controls].astype(float))
    y = df_cl['GMI']

    model = sm.OLS(y, X).fit()
    print(f"Кластер {cl}")
    print(model.summary())

    coef = model.params['PARL']
    p_value = model.pvalues['PARL']
    r_squared = model.rsquared
    std_err = model.bse['PARL']

    results_multi.append({
        'cluster_name': cl,
        'coef': coef,
        'p_value': round(p_value,4),
        'r_squared': r_squared,
        'std_err': std_err
    })

results_multi_df = pd.DataFrame(results_multi)
display(results_multi_df)

#### 8. Множественная регрессия по кластерам (общая) + религия

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

# Подтягиваем кластеры по институтам
clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

# Создаём дамми-переменные по религии
df = pd.get_dummies(df, columns=['Religion'], drop_first=True)

# Переменные модели
base_controls = ['PARL', 'GDP', 'Political Stability', 'Rule of Law',
                 'Fertility', 'Corruption', 'Women Business']

religion_dummies = [col for col in df.columns if col.startswith('Religion_')]

controls = base_controls + religion_dummies

# Берём нужные года и данные без пропусков
df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI'] + controls)

# Модели по кластерам
results_multi = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Конвертируем все X в float
    X = sm.add_constant(df_cl[controls].astype(float))
    y = df_cl['GMI']

    model = sm.OLS(y, X).fit()
    print(f"Кластер {cl}")
    print(model.summary())

    coef = model.params['PARL']
    p_value = model.pvalues['PARL']
    r_squared = model.rsquared
    std_err = model.bse['PARL']

    results_multi.append({
        'cluster_name': cl,
        'coef': coef,
        'p_value': round(p_value,4),
        'r_squared': r_squared,
        'std_err': std_err
    })

results_multi_df = pd.DataFrame(results_multi)
display(results_multi_df)

#### 9. Множественная регрессия по кластерам (общая)

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

# Подтягиваем кластеры по институтам
clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

# Переменные модели
controls = ['PARL', 'GDP', 'Political Stability', 'Rule of Law',
                 'Fertility', 'Corruption', 'Women Business']

# Берём нужные года и данные без пропусков
df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI'] + controls)

# Модели по кластерам
results_multi = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Конвертируем все X в float
    X = sm.add_constant(df_cl[controls].astype(float))
    y = df_cl['GMI']

    model = sm.OLS(y, X).fit()
    print(f"Кластер {cl}")
    print(model.summary())

    coef = model.params['PARL']
    p_value = model.pvalues['PARL']
    r_squared = model.rsquared
    std_err = model.bse['PARL']

    results_multi.append({
        'cluster_name': cl,
        'coef': coef,
        'p_value': round(p_value,4),
        'r_squared': r_squared,
        'std_err': std_err
    })

results_multi_df = pd.DataFrame(results_multi)
display(results_multi_df)

In [ ]:
# График коэффициентов
fig = px.bar(
    results_multi_df,
    x='cluster_name',
    y='coef',
    error_y='std_err',
    title='Коэффициенты PARL в моделях по кластерам (множ. модель + религия)',
    labels={'coef': 'Коэффициент при PARL', 'cluster': 'Кластер'}
)

fig.show()

### 10. Причинность

#### 10.1 Лаговая модель

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

# Подтягиваем кластеры по институтам
clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

# Лагируем PARL
df['PARL_lag'] = df.groupby('Country Name')['PARL'].shift(1)

# Переменные модели
controls = ['PARL_lag', 'GDP', 'Political Stability', 'Rule of Law',
                 'Fertility', 'Corruption', 'Women Business']

# Берём нужные года и данные без пропусков
df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI'] + controls)

# Модели по кластерам
results_multi = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Конвертируем все X в float
    X = sm.add_constant(df_cl[controls].astype(float))
    y = df_cl['GMI']

    model = sm.OLS(y, X).fit()
    print(f"Кластер {cl}")
    print(model.summary())

    # PARL_lag
    coef_lag = model.params['PARL_lag']
    p_value_lag = model.pvalues['PARL_lag']
    std_err_lag = model.bse['PARL_lag']

    r_squared = model.rsquared

    results_multi.append({
        'cluster_name': cl,
        'coef_lag': coef_lag,
        'p_value_lag': round(p_value_lag,4),
        'std_err_lag': std_err_lag,
        'r_squared': r_squared
    })

results_multi_df = pd.DataFrame(results_multi)
display(results_multi_df)

#### 10.2 Тест Грейнджера (PARL влияет на GMI)

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

# Фильтруем на нужный период
df = df[(df['Year'] >= 1997)]

# Итоговый результат
results_gc = []

for cl in sorted(df['cluster_name'].unique()):
    print(f'Кластер: {cl}')
    df_cl = df[df['cluster_name'] == cl]

    for country in df_cl['Country Name'].unique():
        df_c = df_cl[df_cl['Country Name'] == country][['GMI', 'PARL']].dropna()

        if len(df_c) > 5:  # минимум данных для теста
            try:
                test_result = grangercausalitytests(df_c, maxlag=1, verbose=False)
                p_value = test_result[1][0]['ssr_ftest'][1]  # p-value на лаге 1

                results_gc.append({
                    'cluster_name': cl,
                    'country': country,
                    'p_value': round(p_value, 4)
                })

            except Exception as e:
                print(f'Skip {country} — {e}')

gc_results_df = pd.DataFrame(results_gc)

In [ ]:
print(gc_results_df.groupby('cluster_name')['p_value'].apply(lambda x: (x <= 0.05).sum() / len(x)).reset_index(name='share_with_causality'))

#### 10.3 Тест Грейнджера (GMI влияет на PARL)

In [ ]:
results_gc_reverse = []

for cl in sorted(df['cluster_name'].unique()):
    print(f'Кластер: {cl}')
    df_cl = df[df['cluster_name'] == cl]

    for country in df_cl['Country Name'].unique():
        df_c = df_cl[df_cl['Country Name'] == country][['PARL', 'GMI']].dropna()

        if len(df_c) > 5:
            try:
                test_result = grangercausalitytests(df_c, maxlag=1, verbose=False)
                p_value = test_result[1][0]['ssr_ftest'][1]

                results_gc_reverse.append({
                    'cluster_name': cl,
                    'country': country,
                    'p_value': round(p_value, 4)
                })

            except Exception as e:
                print(f'Skip {country} — {e}')

gc_results_reverse_df = pd.DataFrame(results_gc_reverse)

In [ ]:
print(gc_results_reverse_df.groupby('cluster_name')['p_value'].apply(lambda x: (x <= 0.05).sum() / len(x)).reset_index(name='share_with_causality'))

#### 10.4 Тест Грейнджера для России

In [ ]:
# Отбираем данные только по России
df_russia = df[df['Country Name'] == 'Russia'][['GMI', 'PARL']].dropna()

print(f'Число наблюдений по России: {len(df_russia)}')

# Проверяем, достаточно ли данных
if len(df_russia) > 5:
    grangercausalitytests(df_russia, maxlag=1, verbose=True)
else:
    print('Недостаточно данных для теста')

In [ ]:
# Отбираем данные только по России
df_russia = df[df['Country Name'] == 'Russia'][['PARL', 'GMI']].dropna()

print(f'Число наблюдений по России: {len(df_russia)}')

# Проверяем, достаточно ли данных
if len(df_russia) > 5:
    grangercausalitytests(df_russia, maxlag=1, verbose=True)
else:
    print('Недостаточно данных для теста')

#### 10.5 Регрессия с инструментальными переменными (Women  Business)

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

df = df.rename(columns={
    'Political Stability': 'Political_Stability',
    'Rule of Law': 'Rule_of_Law',
    'Women Business': 'Women_Business'
})

clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

controls = ['GDP', 'Political_Stability', 'Rule_of_Law', 'Fertility', 'Corruption']

df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI', 'PARL', 'Women_Business'] + controls)

results_iv = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Первая стадия: PARL ~ Women Business + controls
    first_stage = sm.OLS(df_cl['PARL'], sm.add_constant(df_cl[['Women_Business'] + controls])).fit()

    print(f'Кластер: {cl} — Первая стадия')
    print(first_stage.summary())

    # Вторая стадия: GMI ~ PARL_hat + controls
    iv_model = IV2SLS.from_formula(
        'GMI ~ 1 + ' + ' + '.join(controls) + ' [PARL ~ Women_Business]',
        data=df_cl
    ).fit()

    print(f'Кластер: {cl} — Вторая стадия')
    print(iv_model.summary)

    results_iv.append({
        'cluster_name': cl,
        'coef_parl': iv_model.params['PARL'],
        'p_value_parl': round(iv_model.pvalues['PARL'], 4),
        'r_squared': iv_model.rsquared
    })

results_iv_df = pd.DataFrame(results_iv)
display(results_iv_df)

In [ ]:
controls = ['GDP', 'Political_Stability', 'Rule_of_Law',
            'Fertility', 'Corruption']

df_model = df[(df['Year'] >= 1997)].dropna(subset=['PARL', 'Women_Business'] + controls)

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    X = df_cl[['Women_Business'] + controls]
    X = sm.add_constant(X)
    y = df_cl['PARL']

    model = sm.OLS(y, X).fit()

    print(f'Кластер: {cl}')
    print(model.summary())
    print('\n')

#### 10.6 Регрессия с инструментальными переменными (Female Labor Force)

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

df = df.rename(columns={
    'Political Stability': 'Political_Stability',
    'Rule of Law': 'Rule_of_Law'
})

clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

controls = ['GDP', 'Political_Stability', 'Rule_of_Law', 'Fertility', 'Corruption']

df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI', 'PARL', 'Female_Labor_Force'] + controls)

results_iv = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Первая стадия: PARL ~ Women Business + controls
    first_stage = sm.OLS(df_cl['PARL'], sm.add_constant(df_cl[['Female_Labor_Force'] + controls])).fit()

    print(f'Кластер: {cl} — Первая стадия')
    print(first_stage.summary())

    # Вторая стадия: GMI ~ PARL_hat + controls
    iv_model = IV2SLS.from_formula(
        'GMI ~ 1 + ' + ' + '.join(controls) + ' [PARL ~ Female_Labor_Force]',
        data=df_cl
    ).fit()

    print(f'Кластер: {cl} — Вторая стадия')
    print(iv_model.summary)

    results_iv.append({
        'cluster_name': cl,
        'coef_parl': iv_model.params['PARL'],
        'p_value_parl': round(iv_model.pvalues['PARL'], 4),
        'r_squared': iv_model.rsquared
    })

results_iv_df = pd.DataFrame(results_iv)
display(results_iv_df)

In [ ]:
controls = ['GDP', 'Political_Stability', 'Rule_of_Law',
            'Fertility', 'Corruption']

df_model = df[(df['Year'] >= 1997)].dropna(subset=['PARL', 'Female_Labor_Force'] + controls)

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    X = df_cl[['Female_Labor_Force'] + controls]
    X = sm.add_constant(X)
    y = df_cl['PARL']

    model = sm.OLS(y, X).fit()

    print(f'Кластер: {cl}')
    print(model.summary())
    print('\n')

#### 10.7 Overidentification тест (Women Business и Female Labor Force)

In [ ]:
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

df = df.rename(columns={
    'Political Stability': 'Political_Stability',
    'Rule of Law': 'Rule_of_Law',
    'Women Business': 'Women_Business'
})

clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

controls = ['GDP', 'Political_Stability', 'Rule_of_Law', 'Fertility', 'Corruption']

df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI', 'PARL', 'Women_Business', 'Female_Labor_Force'] + controls)

results_iv = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # IV модель с двумя инструментами
    iv_model = IV2SLS.from_formula(
        'GMI ~ 1 + ' + ' + '.join(controls) + ' [PARL ~ Women_Business + Female_Labor_Force]',
        data=df_cl
    ).fit()

    print(f'Кластер: {cl}')
    print(iv_model.summary)
    print('\n')

    # Hansen J-test (Sargan test)
    overid_p = iv_model.wooldridge_overid.pval

    results_iv.append({
        'cluster_name': cl,
        'coef_parl': iv_model.params['PARL'],
        'p_value_parl': round(iv_model.pvalues['PARL'], 4),
        'r_squared': iv_model.rsquared,
        'overid_p_value': round(overid_p, 4)
    })

results_iv_df = pd.DataFrame(results_iv)
display(results_iv_df)

#### 10.8 Проверяем Female Adult Literacy

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

df = df.rename(columns={
    'Political Stability': 'Political_Stability',
    'Rule of Law': 'Rule_of_Law',
    'Women Business': 'Women_Business'
})

clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

controls = ['GDP', 'Political_Stability', 'Rule_of_Law', 'Fertility', 'Corruption']

df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI', 'PARL', 'Female_Adult_Literacy'] + controls)

results_iv = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Проверяем количество наблюдений
    if len(df_cl) <= len(controls) + 1:
        print(f'Кластер {cl} — слишком мало данных для IV-регрессии, пропускаем')
        continue

    # Первая стадия
    first_stage = sm.OLS(df_cl['PARL'], sm.add_constant(df_cl[['Female_Adult_Literacy'] + controls])).fit()
    print(f'Кластер: {cl} — Первая стадия')
    print(first_stage.summary())

    # Вторая стадия
    iv_model = IV2SLS.from_formula(
        'GMI ~ 1 + ' + ' + '.join(controls) + ' [PARL ~ Female_Adult_Literacy]',
        data=df_cl
    ).fit()

    print(f'Кластер: {cl} — Вторая стадия')
    print(iv_model.summary)

    results_iv.append({
        'cluster_name': cl,
        'coef_parl': iv_model.params['PARL'],
        'p_value_parl': round(iv_model.pvalues['PARL'], 4),
        'r_squared': iv_model.rsquared
    })

results_iv_df = pd.DataFrame(results_iv)
display(results_iv_df)

#### 10.9 Проверяем Women Civil Society Index

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

df = df.rename(columns={
    'Political Stability': 'Political_Stability',
    'Rule of Law': 'Rule_of_Law',
    'Women Business': 'Women_Business'
})

clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

controls = ['GDP', 'Political_Stability', 'Rule_of_Law', 'Fertility', 'Corruption']

df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI', 'PARL', 'Women_Civil_Society_Index'] + controls)

results_iv = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    # Проверяем количество наблюдений
    if len(df_cl) <= len(controls) + 1:
        print(f'Кластер {cl} — слишком мало данных для IV-регрессии, пропускаем')
        continue

    # Первая стадия
    first_stage = sm.OLS(df_cl['PARL'], sm.add_constant(df_cl[['Women_Civil_Society_Index'] + controls])).fit()
    print(f'Кластер: {cl} — Первая стадия')
    print(first_stage.summary())

    # Вторая стадия
    iv_model = IV2SLS.from_formula(
        'GMI ~ 1 + ' + ' + '.join(controls) + ' [PARL ~ Women_Civil_Society_Index]',
        data=df_cl
    ).fit()

    print(f'Кластер: {cl} — Вторая стадия')
    print(iv_model.summary)

    results_iv.append({
        'cluster_name': cl,
        'coef_parl': iv_model.params['PARL'],
        'p_value_parl': round(iv_model.pvalues['PARL'], 4),
        'r_squared': iv_model.rsquared
    })

results_iv_df = pd.DataFrame(results_iv)
display(results_iv_df)

#### 10.10 Overidentification тест (Women Business и Women Civil Society Index)

In [ ]:
# Загружаем данные
df = pd.read_csv('phd_data/final_data.csv', sep=';', decimal=',')

# Переименуем колонки для удобства
df = df.rename(columns={
    'Political Stability': 'Political_Stability',
    'Rule of Law': 'Rule_of_Law',
    'Women Business': 'Women_Business',
    'Women Civil Society Participation': 'Women_Civil_Society_Index'
})

# Добавляем кластеры
clusters = pd.read_csv('country_clusters.csv')
df = df.merge(clusters, on='Country Name')

# Контрольные переменные
controls = ['GDP', 'Political_Stability', 'Rule_of_Law', 'Fertility', 'Corruption']

# Фильтруем релевантные данные
df_model = df[(df['Year'] >= 1997)].dropna(subset=['GMI', 'PARL', 'Women_Business', 'Women_Civil_Society_Index'] + controls)

results_iv = []

for cl in sorted(df_model['cluster_name'].unique()):
    df_cl = df_model[df_model['cluster_name'] == cl]

    iv_model = IV2SLS.from_formula(
        'GMI ~ 1 + ' + ' + '.join(controls) + ' [PARL ~ Women_Business + Women_Civil_Society_Index]',
        data=df_cl
    ).fit()

    print(f'Кластер: {cl}')
    print(iv_model.summary)
    print('\n')

    # Hansen's J-test (проверка валидности обоих инструментов вместе)
    overid_p = iv_model.wooldridge_overid.pval

    results_iv.append({
        'cluster_name': cl,
        'coef_parl': iv_model.params['PARL'],
        'p_value_parl': round(iv_model.pvalues['PARL'], 4),
        'r_squared': iv_model.rsquared,
        'overid_p_value': round(overid_p, 4)
    })

results_iv_df = pd.DataFrame(results_iv)
display(results_iv_df)
